# Ontology-Constrained Memory (OCM) — Google Colab runner

A write-time **governed** memory layer for long-horizon LLM agents. This notebook runs the project end to end:

Sections 1–5 run on a **CPU-only** runtime. The section-6 cells need a **GPU runtime** (Runtime → Change runtime type → GPU). Qwen2.5-14B-Instruct in bf16 (~28 GB) fits on a single **A100 40GB**.

## 1. Get the code

Downloads and unpacks the zip. Datasets are not in the archive; fetch them with section 3 of the README.

In [ ]:
# Fetch the anonymized snapshot. Anonymous GitHub exposes no Git remote, so this
# downloads the zip archive instead of cloning.
import io, os, zipfile, urllib.request, urllib.error

REPO_ID = os.environ.get("OCM_REPO_ID", "ocmr-A82F")
ZIP_URL = os.environ.get(
    "OCM_REPO_ZIP", f"https://anonymous.4open.science/api/repo/{REPO_ID}/zip"
)
REPO_DIR = "/content/ocmr"

if os.path.isdir(os.path.join(REPO_DIR, "ocm")):
    print(f"{REPO_DIR} already populated - delete it to re-download.")
else:
    print("downloading", ZIP_URL)
    try:
        req = urllib.request.Request(ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=180) as resp:
            blob = resp.read()
    except urllib.error.HTTPError as exc:
        raise SystemExit(
            f"HTTP {exc.code} from Anonymous GitHub. Confirm the snapshot is still "
            "published and unexpired by opening the /r/<id> page in a browser. "
            "Alternatively set OCM_REPO_ZIP to any reachable zip URL, or upload a "
            "zip to /content and set OCM_REPO_ZIP to its file:// path."
        ) from None
    print(f"  {len(blob) / 1e6:.1f} MB")
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        zf.extractall(REPO_DIR)
    # The archive wraps everything in a single top-level folder; flatten it.
    kids = [k for k in os.listdir(REPO_DIR) if not k.startswith(".")]
    if len(kids) == 1 and os.path.isdir(os.path.join(REPO_DIR, kids[0])):
        inner = os.path.join(REPO_DIR, kids[0])
        for name in os.listdir(inner):
            os.replace(os.path.join(inner, name), os.path.join(REPO_DIR, name))
        os.rmdir(inner)

%cd {REPO_DIR}
print("cwd:", os.getcwd())
assert os.path.isdir("ocm"), "unpacked tree has no ocm/ package - check the snapshot"
print("NOTE: if the kernel already imported ocm.*, Restart the runtime now.")

## 2. Install dependencies

The core (offline) demo needs only a few light packages. `chromadb` and `sentence-transformers` are **optional** — OCM falls back to a pure-Python vector index and a deterministic embedding provider when they are absent.

In [ ]:
# Light core install (enough for sections 2–4).
!pip -q install "pydantic>=2.6,<3" "networkx>=3.2" "fastapi>=0.110" "uvicorn>=0.29" "httpx>=0.27" "pytest>=8.0" "hypothesis>=6.100"

import sys, os
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import ocm
print("OCM importable from", ROOT)

## 2b. Persistent output (Google Drive)

Mount Drive so results **and per-(method, seed) checkpoints** survive a Colab refresh/crash — a resumed run skips already-finished work instead of restarting. Set `USE_DRIVE = False` to keep outputs only on the (ephemeral) Colab disk.

In [ ]:
import os
USE_DRIVE = True
OUTPUT_DIR = "/content/ocm_results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/ocm_results"
    except Exception as e:
        print("Drive mount failed; using local dir:", e)
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Outputs ->", OUTPUT_DIR)
print("Checkpoints ->", CKPT_DIR)

## 3. Offline governance demo (no GPU, no API key)

Facts are accepted, a status flip is **quarantined** (not silently overwritten), a correction **supersedes**, and the status query surfaces the contradiction inline.

In [ ]:
from ocm.core.config import Settings
from ocm.core.container import CoreContainer

c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory", extractor="mock"))

def show(r, label):
    print(f"\n# {label}")
    print("  accepted   :", [o.candidate.predicate for o in r.accepted])
    print("  superseded :", [o.candidate.predicate for o in r.superseded])
    print("  quarantined:", [o.reason for o in r.quarantined])

show(c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1"), "W1 ownership + assignment")
show(c.write_pipeline.run("Bob completed Task T1.", "s2"), "W2 completion -> T1 done")
show(c.write_pipeline.run("Task T1 is not started.", "s3"), "W3 status flip -> QUARANTINED")
show(c.write_pipeline.run("Actually, Carol is assigned to Task T1.", "s4"), "W4 correction -> SUPERSEDE")

pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("\nQuery: 'What is the current status of Task T1?'")
print("  answer   :", pkg.answer)
print("  conflicts:", [{"accepted": cf.accepted, "quarantined": cf.quarantined, "reason": cf.reason} for cf in pkg.conflicts])

## 4. Full experiment suite — offline reference run (multi-seed CIs, significance, τ-sweep, stress)

Runs the **full research protocol** (5 seeds × the full benchmark) on the offline mock extractor — a fast, deterministic reference. The genuine LLM-driven run is **Section 6b** (Qwen + real embeddings).

In [ ]:
!python -m ocm.scripts.run_experiments --seeds 1337 7 42 99 2024 \
  --checkpoint-dir {CKPT_DIR}/offline --out {OUTPUT_DIR}/results_offline.json

# Resumable: re-run after a crash/refresh and it skips finished (method, seed) work.
# This is the CPU-only reference run (offline mock extractor + deterministic
# embeddings). For the *real* LLM-driven research run (Qwen2.5-14B-Instruct +
# real embeddings) see Section 6b.
#
# Fast smoke run instead:  !python -m ocm.scripts.run_experiments --quick

## 5. Real embeddings

Swaps the deterministic hashing embeddings for the real `all-MiniLM-L6-v2` model (downloads ~90 MB on first run). Runs on CPU or GPU.

In [ ]:
!pip -q install "sentence-transformers>=2.6"

from ocm.core.config import Settings
from ocm.core.container import CoreContainer

s = Settings(deterministic_test_mode=False, extractor="mock", embedding_mode="local",
             sqlite_path=":memory:", chroma_mode="memory")
c = CoreContainer(s)
c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1")
print("owner answer:", c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5).answer)

## 6. Local Qwen extractor — in-process via `transformers`

Loads **`Qwen/Qwen2.5-14B-Instruct`** in the same process (no server, no vLLM) and plugs it into OCM as the W1 extractor via `TransformersExtractor`.

Loaded in **full bf16** (~28 GB) so it runs fully on-GPU on a single **A100 40GB** (the common Colab A100) — full precision and fast. The load cell has commented alternatives (4-bit 32B for 40GB, or full bf16 32B for 80GB). If generations take minutes, the model is offloading to CPU — check the `offloaded modules` print.

In [ ]:
!pip -q install "transformers>=4.45" accelerate  # add `bitsandbytes` only for the 4-bit 32B alternative

In [ ]:
# Load Qwen2.5-14B-Instruct in full bf16 (~28 GB) — fits fully on-GPU on a
# single A100 40GB (no CPU offload, full precision, fast).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Loaded", MODEL_ID, "(bf16)")

# Sanity check: NOTHING should be on cpu/disk. If it is, the model is offloading
# (the cause of multi-minute generations) — use a smaller model or more VRAM.
dm = getattr(llm_model, "hf_device_map", {})
offloaded = [k for k, v in dm.items() if v in ("cpu", "disk")]
print("offloaded modules:", offloaded or "none (all on GPU \u2713)")

# --- Alternatives -------------------------------------------------------------
# Qwen2.5-32B-Instruct in 4-bit (NF4, ~20 GB) — fits on A100 40GB, larger model
# but quantized (document it in the paper). Needs: pip install bitsandbytes
# from transformers import BitsAndBytesConfig
# MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
# llm_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.bfloat16, device_map="auto")
# Qwen2.5-32B-Instruct in full bf16 (~64 GB) — needs an A100 80GB:
# MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
# llm_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

In [ ]:
# Plug the local model into OCM as the W1 extractor and run the governed pipeline.
from ocm.extraction.transformers_extractor import TransformersExtractor
from ocm.core.container import CoreContainer
from ocm.core.config import Settings

extractor = TransformersExtractor(model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=1024)
# deterministic embeddings + in-memory storage keep everything else hermetic; the LLM does W1.
c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory"), extractor=extractor)

for t, label in [
    ("Alice owns Project Orion. Bob is assigned to Task T1.", "W1"),
    ("Bob completed Task T1.", "W2"),
    ("Task T1 is not started.", "W3"),
]:
    r = c.write_pipeline.run(t, label)
    print(label, "| accepted", [o.candidate.predicate for o in r.accepted],
          "| quarantined", [bool(o.reason) for o in r.quarantined])

pkg = c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5)
print("owner:", pkg.answer)
pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("status:", pkg.answer, "| conflicts:", [(cf.accepted, cf.quarantined) for cf in pkg.conflicts])

### 6b. Full research experiment — Qwen2.5-14B-Instruct + real embeddings

**This is the real research run.** It executes the entire protocol (baselines B0–B3, `Bsup`, the extended comparison arms, the ablations, 5 seeds, the full benchmark, τ-sweep, stress) driven by the **local Qwen extractor** with **real `all-MiniLM-L6-v2` embeddings**. The model + embeddings are loaded **once** and shared across every arm.

Requires Section 5 (real embeddings installed) and Section 6 (Qwen loaded). **At full scale on a single A100 40GB this can take a few hours** — shrink `SEEDS` / `PER_CATEGORY` for a faster pass.


In [ ]:
import logging, os
from transformers import set_seed
from ocm.evaluation import experiment as exp
from ocm.extraction.transformers_extractor import TransformersExtractor
from ocm.extraction.caching_extractor import CachingExtractor
from ocm.retrieval.embeddings import LocalEmbeddingProvider
from ocm.evaluation.run_identity import run_fingerprint

logging.getLogger("ocm").setLevel(logging.ERROR)  # quiet expected governance warnings

set_seed(1337)

_base_extractor = TransformersExtractor(
    model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=1024
)
RUN_FP = run_fingerprint(extractor=_base_extractor, embeddings=None)
print("run fingerprint:", RUN_FP)
qwen_extractor = CachingExtractor(
    _base_extractor,
    cache_path=os.path.join(CKPT_DIR, "qwen", f"extract_cache__{RUN_FP}.json"),
)
if getattr(qwen_extractor, "unversioned_cache", False):
    print(
        "WARNING: extraction cache carries no fingerprint (legacy format); "
        "do not cite this run as reproducible paper evidence."
    )
# REAL all-MiniLM-L6-v2 embeddings (Section 5 installed it). NOTE: do NOT set
# deterministic_test_mode here — that would swap in fake hashing embeddings and
# disable the R2 semantic retriever. The full run uses real embeddings.
real_embeddings = LocalEmbeddingProvider()

# Real token counts for Table V's token-overhead column (uses the Qwen
# tokenizer; falls back to a whitespace proxy if omitted).
token_counter = lambda s: llm_tokenizer.encode(s)

SEEDS = [1337, 7, 42, 99, 2024]   # full protocol; reduce (e.g. [1337]) to shorten
PER_CATEGORY = 25                  # full benchmark; reduce (e.g. 5) to shorten

# Baselines: canonical B0-B3 + supersession-only ablation + extended comparisons.
#   Bsup = latest-value supersession only, without ontology/provenance/temporal governance
#   Brag = RAG-only (vectors-only retrieval, answer from text, no governance)
#   Brtcf = retrieval-time contradiction filter (no write gate; filter at read)
BASELINES = ("B0", "B1", "B2", "Bsup", "B3", "Brag", "Brtcf")

report = exp.run_full_suite(
    seeds=SEEDS,
    per_category=PER_CATEGORY,
    baselines=BASELINES,
    tau=0.8,
    stress_per_class=30,
    extractor=qwen_extractor,
    embeddings=real_embeddings,
    token_counter=token_counter,
    run_fingerprint=RUN_FP,   # separates checkpoints by extraction stack
    checkpoint_dir=os.path.join(CKPT_DIR, "qwen"),          # resume on crash/refresh
    out_path=os.path.join(OUTPUT_DIR, "results_qwen.json"),  # final report on Drive
)
exp.print_report(report)
qwen_extractor.save()  # persist the extraction cache to Drive
print("extraction cache:", qwen_extractor.stats)
print("\nSaved ->", report.get("_saved_to"))


### 6c. Governed-write replay — qualitative evidence + false-quarantine reconciliation
Replays the benchmark through the full governed write path and dumps real accepted / superseded / quarantined examples, plus the false-quarantine reconciliation (shared-store protocol vs per-example isolation). Reuses the **same cached Qwen extractor**, so it adds no new LLM calls beyond the cache. Writes `governance_examples.json` to Drive.

In [ ]:
# Governed-write replay: real governance examples + false-quarantine reconciliation.
from ocm.evaluation.replay_governed_writes import replay_governed_writes

# Shared-store protocol (the harness default): all examples in one governed store.
gov_shared = replay_governed_writes(
    seeds=SEEDS, per_category=PER_CATEGORY,
    extractor=qwen_extractor, embeddings=real_embeddings,
    isolate_per_example=False,
    out_path=os.path.join(OUTPUT_DIR, "governance_examples.json"),
)
# Per-example isolation: removes cross-example identifier collisions, so residual
# quarantines are only within-example (the true false-quarantine floor).
gov_isolated = replay_governed_writes(
    seeds=SEEDS, per_category=PER_CATEGORY,
    extractor=qwen_extractor, embeddings=real_embeddings,
    isolate_per_example=True,
    verbose=False,
)
qwen_extractor.save()
fq_s, q_s = gov_shared["false_quarantine_total"], gov_shared["totals"]["quarantined"]
fq_i, q_i = gov_isolated["false_quarantine_total"], gov_isolated["totals"]["quarantined"]
print(f"\nFalse-quarantine (shared):   {fq_s} of {q_s} quarantines "
      f"({100.0*fq_s/q_s:.1f}%)" if q_s else "no quarantines")
print(f"False-quarantine (isolated): {fq_i} of {q_i} quarantines "
      f"({100.0*fq_i/q_i:.1f}%)" if q_i else "no quarantines")
print("Saved ->", os.path.join(OUTPUT_DIR, "governance_examples.json"))


### 6d. Real-data validation — MultiWOZ 2.2 (governed vs ungoverned)
Maps MultiWOZ dialogue-state slots onto governed single-valued memory (`Slot -[HAS_VALUE]-> SlotValue`, 1:1) via an **oracle extractor** that replays the gold per-turn belief state. Governance is evaluated *given* correct slots (isolating it from dialogue-state-tracking error). Headline: the governed arm (B3) supersedes a changed slot — eliminating durable constraint violations with no recall cost — while ungoverned arms (B0/B2) keep both values. Needs `datasets`; confirm the MultiWOZ license.

In [ ]:
# 6d. MultiWOZ 2.2 real-data run. Oracle extraction => no LLM calls (fast).
# Self-contained: puts the repo on sys.path and falls back to local result dirs,
# so it runs after a restart even if only this cell is executed.
import os, sys, json
REPO_DIR = "/content/ocmr"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ocm.evaluation.datasets.multiwoz_adapter import load_multiwoz, run_multiwoz_suite
from ocm.retrieval.embeddings import LocalEmbeddingProvider
from ocm.evaluation.run_identity import run_fingerprint


SEEDS = globals().get("SEEDS", [1337, 7, 42, 99, 2024])
real_embeddings = globals().get("real_embeddings") or LocalEmbeddingProvider()
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/content/ocm_results")
CKPT_DIR = globals().get("CKPT_DIR", os.path.join(OUTPUT_DIR, "checkpoints"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(CKPT_DIR, "qwen"), exist_ok=True)

# MultiWOZ 2.2 is fetched from the official GitHub JSON (stdlib HTTP); no install.
# limit=None for the full validation (dev) split; set e.g. 300 for a quick pass.
MWZ_LIMIT = None
dialogues = load_multiwoz(split="validation", limit=MWZ_LIMIT)
print(f"loaded {len(dialogues)} MultiWOZ dialogues")

MWZ_REUSE_LEGACY_CHECKPOINTS = bool(
    globals().get("MWZ_REUSE_LEGACY_CHECKPOINTS", False)
)
if MWZ_REUSE_LEGACY_CHECKPOINTS:
    MWZ_FP = None
    print("multiwoz: reusing legacy checkpoint keys (no __fp fragment)")
else:
    MWZ_FP = run_fingerprint(
        embeddings=real_embeddings,
        dataset="multiwoz_2.2",
        split="validation",
        n_dialogues=len(dialogues),
    )
    print("multiwoz fingerprint:", MWZ_FP)

mwz_report = run_multiwoz_suite(
    dialogues,
    baselines=("B0", "B2", "Bsup", "B3"),
    run_fingerprint=MWZ_FP,
    seeds=SEEDS,
    embeddings=real_embeddings,
    checkpoint_dir=os.path.join(CKPT_DIR, "qwen"),
)

print("\n=== MultiWOZ decisive metrics (mean [95% CI]) ===")
print(f"{'Method':<8}{'TaskSuccess up':<22}{'Contradiction dn':<22}{'ConstraintViol dn':<22}")
for m in mwz_report["methods"]:
    d = mwz_report["decisive_metrics"][m]
    def _ci(k):
        x = d[k]; return f"{x['mean']:.1f} [{x['low']:.1f},{x['high']:.1f}]"
    print(f"{m:<8}{_ci('task_success'):<22}{_ci('contradiction_rate'):<22}{_ci('constraint_violations'):<22}")
print("\nwrite outcomes:", {m: mwz_report["write_outcomes"][m] for m in mwz_report["methods"]})

_mwz_path = os.path.join(OUTPUT_DIR, "results_multiwoz.json")
with open(_mwz_path, "w") as fh:
    json.dump(mwz_report, fh, indent=2, default=str)
print("Saved ->", _mwz_path)


### 6e. Real-data validation #2 — LongMemEval knowledge-updates (governed vs ungoverned)

Generalizes the MultiWOZ finding to **open-domain** chat on a recognized agent-memory benchmark (LongMemEval, ICLR 2025). A user fact is stated then *changed* across sessions; the governed arm (B3) **supersedes** the stale value (current value retrievable, zero durable violations) while ungoverned arms keep both. **Arm A (oracle):** a one-time Qwen annotation pass extracts the gold value trajectory per `knowledge-update` question (validated against the benchmark answer), cached to Drive; governance is then evaluated *given* gold facts (isolating it from extraction error, exactly like 6d). Also scores governed **abstention**. Requires Section 6 (Qwen loaded) for the annotation pass only; the cached annotations make re-runs LLM-free.

In [ ]:
# 6e. LongMemEval knowledge-update real-data run (Arm A / oracle).
# Self-contained: repo on sys.path + fallback dirs, so it runs after a restart.
# The one-time annotation pass reuses the Section-7 Qwen (llm_model/llm_tokenizer);
# once cached to Drive the suite itself is LLM-free (oracle extraction).
import os, sys, json, urllib.request
REPO_DIR = "/content/ocmr"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ocm.evaluation.datasets.longmemeval_adapter import (
    load_longmemeval, run_longmemeval_suite, evaluate_abstention,
)
from ocm.evaluation.datasets.longmemeval_annotate import annotate_file, load_annotations
from ocm.retrieval.embeddings import LocalEmbeddingProvider
from ocm.evaluation.run_identity import json_digest, run_fingerprint

SEEDS = globals().get("SEEDS", [1337, 7, 42, 99, 2024])
real_embeddings = globals().get("real_embeddings") or LocalEmbeddingProvider()
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/content/ocm_results")
CKPT_DIR = globals().get("CKPT_DIR", os.path.join(OUTPUT_DIR, "checkpoints"))
DATA_DIR = os.path.join(REPO_DIR, "data") if os.path.isdir(REPO_DIR) else "/content/data"
for d in (OUTPUT_DIR, os.path.join(CKPT_DIR, "qwen"), DATA_DIR):
    os.makedirs(d, exist_ok=True)

# 1) LongMemEval data (oracle file = evidence sessions only; small, ~public HF).
LME_PATH = os.path.join(DATA_DIR, "longmemeval_oracle.json")
if not os.path.exists(LME_PATH):
    url = "https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_oracle.json"
    print("downloading", url)
    urllib.request.urlretrieve(url, LME_PATH)
print("LongMemEval data:", LME_PATH)

# 2) One-time gold-trajectory annotation (cached to Drive; reuses loaded Qwen).
#    The trajectories are LLM-generated, so the filename records WHICH model
#    produced them. Reusing 6B-annotated gold beside a 14B run would silently mix
#    two configurations, which is the failure mode that made the earlier Arm-B
#    artifacts uncitable. Set ANN_MODEL explicitly to load a prebuilt file.
ANN_MODEL = (
    globals().get("ANN_MODEL")
    or globals().get("MODEL_ID")
    or "unknown-model"
)
_ann_slug = str(ANN_MODEL).replace("/", "_").replace(" ", "_")
ANN_PATH = os.path.join(
    OUTPUT_DIR, f"longmemeval_kupdate_annotations__{_ann_slug}.json"
)
_ANN_LEGACY = os.path.join(OUTPUT_DIR, "longmemeval_kupdate_annotations.json")
print("annotation cache:", ANN_PATH)
if os.path.exists(ANN_PATH):
    annotations = load_annotations(ANN_PATH)
    print(f"loaded {len(annotations)} cached annotations for {ANN_MODEL}")
elif os.path.exists(_ANN_LEGACY):
    # Don't strand existing work, but don't pretend we know what made it either.
    annotations = load_annotations(_ANN_LEGACY)
    print(
        f"WARNING: loaded {len(annotations)} annotations from the UNSTAMPED file\n"
        f"  {_ANN_LEGACY}\n"
        "  The annotating model is unrecorded. If these were produced by a "
        "different model than the current run, do not cite the result. Rename the "
        "file to the stamped path above once you know its provenance."
    )
else:
    if "llm_model" not in globals() or "llm_tokenizer" not in globals():
        raise RuntimeError(
            "Annotation pass needs the Qwen model from Section 6 "
            "(run the model load cell first), or copy a prebuilt "
            f"annotations file to {ANN_PATH}."
        )
    import torch
    def qwen_chat(prompt: str) -> str:
        msgs = [{"role": "system", "content": "You label data and output only a JSON object."},
                {"role": "user", "content": prompt}]
        text = llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)
        with torch.no_grad():
            out = llm_model.generate(**inputs, max_new_tokens=256, do_sample=False)
        gen = out[0][inputs["input_ids"].shape[1]:]
        return llm_tokenizer.decode(gen, skip_special_tokens=True)
    annotations = annotate_file(LME_PATH, ANN_PATH, qwen_chat)
    print(f"annotated + validated {len(annotations)} knowledge-update questions -> {ANN_PATH}")

# 3) Knowledge-update governed vs ungoverned (oracle arm). Checkpoint suffix
#    __lme_kupdate keeps these separate from synthetic + MultiWOZ checkpoints.
ku_instances = load_longmemeval(LME_PATH, question_type="knowledge-update")
print(f"{len(ku_instances)} knowledge-update questions; {len(annotations)} annotated")
# Checkpoint identity for this arm. The suite replay is oracle-driven, so it does
# not depend on the loaded extractor; it depends on the embedding stack and on the
# gold annotations. Digesting the annotations captures the annotating model's
# output exactly: regenerate them under a different model and these checkpoints
# recompute; leave them unchanged and they resume.
LME_FP = run_fingerprint(
    embeddings=real_embeddings,
    dataset="longmemeval_kupdate",
    annotating_model=str(ANN_MODEL),
    annotations_sha=json_digest(annotations, length=16),
)
print("longmemeval Arm-A fingerprint:", LME_FP)

lme_report = run_longmemeval_suite(
    ku_instances, annotations,
    baselines=("B0", "B2", "Bsup", "B3"), seeds=SEEDS,
    run_fingerprint=LME_FP,
    embeddings=real_embeddings,
    checkpoint_dir=os.path.join(CKPT_DIR, "qwen"),
)
print("\n=== LongMemEval knowledge-update decisive metrics (mean [95% CI]) ===")
print(f"{'Method':<8}{'TaskSuccess up':<22}{'Contradiction dn':<22}{'ConstraintViol dn':<22}")
for m in lme_report["methods"]:
    d = lme_report["decisive_metrics"][m]
    def _ci(k):
        x = d[k]; return f"{x['mean']:.1f} [{x['low']:.1f},{x['high']:.1f}]"
    print(f"{m:<8}{_ci('task_success'):<22}{_ci('contradiction_rate'):<22}{_ci('constraint_violations'):<22}")
print("\nwrite outcomes:", {m: lme_report["write_outcomes"][m] for m in lme_report["methods"]})

# 4) Abstention (oracle-mode plumbing metric; cross-baseline divergence is an
#    end-to-end result, see the adapter docstring).
abst_instances = load_longmemeval(LME_PATH, question_type=None, abstention=True)
abst = evaluate_abstention(abst_instances, baselines=("B0", "B2", "Bsup", "B3"),
                           embeddings=real_embeddings)
print(f"\nabstention accuracy (n={len(abst_instances)}):", abst)

_p = os.path.join(OUTPUT_DIR, "results_longmemeval.json")
with open(_p, "w") as fh:
    json.dump({"knowledge_update": lme_report, "abstention": abst}, fh, indent=2, default=str)
print("Saved ->", _p)


### 6f. Real-data validation #2 (end-to-end) — LongMemEval knowledge-updates + abstention from raw text

The honest counterpart to 6e: **no oracle**. The real Qwen extractor reads the full multi-session haystack (`longmemeval_s.json`, ~40 sessions / 115k tokens each) and emits `Slot -[HAS_VALUE]-> SlotValue` candidates; governance then acts on noisy, real extractions. This closes the "but you used gold facts" critique: knowledge-update measures noisy extraction/entity-resolution cost, while `_abs` abstention measures whether the system refuses when the full haystack contains no grounded answer.

The paper rerun defaults below use the full split, the LongMemEval-specific extraction prompt, Qwen slot linking at `0.75`, and the `Bsup` ablation. Extraction and slot linking are cached separately to Drive with identity metadata (dataset hash, prompt, model, max-token budget, decoding config, and code revision), and evaluation checkpoints include the extracted-example fingerprint, so incompatible runs cannot silently reuse each other's artifacts. Requires Section 6 (Qwen loaded).


In [ ]:
# 6f. LongMemEval END-TO-END (Arm B): real extraction from text.
# Self-contained; reuses the Section-7 Qwen as the fact extractor/linker. The
# paper defaults run the full LongMemEval splits and include Bsup.
import os, sys, json, urllib.request
from pathlib import Path
REPO_DIR = "/content/ocmr"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ocm.evaluation.datasets.longmemeval_adapter import (
    load_longmemeval, run_longmemeval_e2e, evaluate_abstention_e2e,
    FACT_EXTRACTION_PROMPTS, build_fact_extract_fn, build_slot_link_fn,
)
from ocm.retrieval.embeddings import LocalEmbeddingProvider
from run_6f_local import (
    CachedChat, SYSTEM_PROMPT, _file_sha256, _git_diff_sha256, _git_revision,
    _json_digest, _safe_segment, _text_sha256,
)

SEEDS = globals().get("SEEDS", [1337, 7, 42, 99, 2024])
real_embeddings = globals().get("real_embeddings") or LocalEmbeddingProvider()
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/content/ocm_results")
CKPT_DIR = globals().get("CKPT_DIR", os.path.join(OUTPUT_DIR, "checkpoints"))
DATA_DIR = os.path.join(REPO_DIR, "data") if os.path.isdir(REPO_DIR) else "/content/data"
for d in (OUTPUT_DIR, os.path.join(CKPT_DIR, "qwen"), os.path.join(CKPT_DIR, "qwen_e2e"), DATA_DIR):
    os.makedirs(d, exist_ok=True)

# Published paper rerun defaults. Override these globals above the cell for a
# smoke run, e.g. LME_E2E_LIMIT=30, LME_ABST_LIMIT=30, SLOT_LINKER="none".
LME_E2E_LIMIT = globals().get("LME_E2E_LIMIT", None)
LME_ABST_LIMIT = globals().get("LME_ABST_LIMIT", None)
INTENT_MODE = globals().get("INTENT_MODE", "auto")
EXTRACT_PROMPT = globals().get("EXTRACT_PROMPT", "longmemeval")
SLOT_LINKER = globals().get("SLOT_LINKER", "qwen")
LINK_THRESHOLD = globals().get("LINK_THRESHOLD", 0.75)
BASELINES_6f = tuple(globals().get("BASELINES_6f", ("B0", "B2", "Bsup", "B3")))

LME_E2E_MAX_NEW_TOKENS = globals().get("LME_E2E_MAX_NEW_TOKENS", 1024)
ALLOW_OFFLOAD = globals().get("ALLOW_OFFLOAD", False)
FLUSH_EVERY = int(globals().get("FLUSH_EVERY", 50))
LEGACY_CACHE_KEYS = bool(globals().get("LEGACY_CACHE_KEYS", False))

# 1) Full-haystack file (sessions + distractors; ~115k tokens each).
LME_S_PATH = os.path.join(DATA_DIR, "longmemeval_s.json")
if not os.path.exists(LME_S_PATH):
    url = "https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json"
    print("downloading", url)
    urllib.request.urlretrieve(url, LME_S_PATH)
print("LongMemEval (full haystack):", LME_S_PATH)

# 2) Real Qwen fact extractor + optional Qwen slot linker with separate,
# identity-stamped caches.
if "llm_model" not in globals() or "llm_tokenizer" not in globals():
    raise RuntimeError("Arm B needs the Section-7 Qwen (run a model load cell first).")
import torch
if EXTRACT_PROMPT not in FACT_EXTRACTION_PROMPTS:
    raise ValueError("EXTRACT_PROMPT must be one of: " + ", ".join(sorted(FACT_EXTRACTION_PROMPTS)))
if SLOT_LINKER not in ("none", "deterministic", "qwen"):
    raise ValueError("SLOT_LINKER must be 'none', 'deterministic', or 'qwen'")

dm = getattr(llm_model, "hf_device_map", {}) or {}
offloaded = [k for k, v in dm.items() if v in ("cpu", "disk")]
print("offloaded modules:", offloaded or "none (all on GPU)")
if offloaded and not ALLOW_OFFLOAD:
    raise RuntimeError("Model offloaded to CPU/disk. Use a smaller model, more VRAM, or set ALLOW_OFFLOAD=True.")

def _limit_segment(value) -> str:
    return "full" if value is None else f"limit_{value}"

def _model_id() -> str:
    config = getattr(llm_model, "config", None)
    for attr in ("_name_or_path", "name_or_path"):
        value = getattr(config, attr, None) or getattr(llm_model, attr, None)
        if value:
            return str(value)
    return globals().get("MODEL_ID", "unknown")

dataset_sha256 = _file_sha256(Path(LME_S_PATH), length=None)
extract_prompt_sha256 = _text_sha256(FACT_EXTRACTION_PROMPTS[EXTRACT_PROMPT], length=None)
code_revision = _git_revision(Path(REPO_DIR))
code_diff_sha256 = _git_diff_sha256(Path(REPO_DIR))
model_id = _model_id()
cache_base_identity = {
    "dataset": "longmemeval_s",
    "dataset_sha256": dataset_sha256,
    "extract_backend": "transformers",
    "llm_model": model_id,
    "llm_max_tokens": LME_E2E_MAX_NEW_TOKENS,
    "decoding": {"temperature": 0, "do_sample": False},
    "system_prompt_sha256": _text_sha256(SYSTEM_PROMPT, length=None),
    "code_revision": code_revision,
    "code_diff_sha256": code_diff_sha256,
}
extract_cache_identity = {
    **cache_base_identity,
    "task": "fact_extraction",
    "extract_prompt": EXTRACT_PROMPT,
    "extract_prompt_sha256": extract_prompt_sha256,
}
link_cache_identity = {
    **cache_base_identity,
    "task": "slot_linking",
    "slot_linker": SLOT_LINKER,
    "slot_link_prompt": "longmemeval-slot-link-v1",
}
cache_suffix = "" if EXTRACT_PROMPT == "durable" else f"_{_safe_segment(EXTRACT_PROMPT)}"
extract_cache_id = _json_digest(extract_cache_identity, length=12)
link_cache_id = _json_digest(link_cache_identity, length=12)
if LEGACY_CACHE_KEYS:
    default_extract_cache = os.path.join(OUTPUT_DIR, f"lme_e2e_extract_cache{cache_suffix}.json")
    default_link_cache = os.path.join(OUTPUT_DIR, f"lme_e2e_link_cache{cache_suffix}.json")
else:
    default_extract_cache = os.path.join(OUTPUT_DIR, f"lme_e2e_extract_cache{cache_suffix}__{extract_cache_id}.json")
    default_link_cache = os.path.join(OUTPUT_DIR, f"lme_e2e_link_cache{cache_suffix}__{link_cache_id}.json")
_EXTRACT_CACHE_PATH = globals().get("EXTRACT_CACHE_PATH", default_extract_cache)
_LINK_CACHE_PATH = globals().get("LINK_CACHE_PATH", default_link_cache)

link_segment = ""
if SLOT_LINKER != "none":
    link_segment = f"__link_{_safe_segment(SLOT_LINKER)}_t{int(round(LINK_THRESHOLD * 100))}"
run_identity = {
    "dataset": "longmemeval_s",
    "dataset_sha256": dataset_sha256,
    "extract_prompt": EXTRACT_PROMPT,
    "extract_prompt_sha256": extract_prompt_sha256,
    "extract_backend": "transformers",
    "llm_model": model_id,
    "llm_max_tokens": LME_E2E_MAX_NEW_TOKENS,
    "slot_linker": SLOT_LINKER,
    "link_threshold": LINK_THRESHOLD,
    "intent_mode": INTENT_MODE,
    "knowledge_update_limit": LME_E2E_LIMIT,
    "abstention_limit": LME_ABST_LIMIT,
    "baselines": BASELINES_6f,
    "seeds": SEEDS,
    "embeddings": type(real_embeddings).__name__,
    "manager": "governed",
    "code_revision": code_revision,
    "code_diff_sha256": code_diff_sha256,
    "legacy_cache_keys": LEGACY_CACHE_KEYS,
}
run_fingerprint = _json_digest(run_identity, length=12)
run_segment = (
    f"extract_{EXTRACT_PROMPT}__intent_{INTENT_MODE}"
    f"__ku_{_limit_segment(LME_E2E_LIMIT)}"
    f"__abs_{_limit_segment(LME_ABST_LIMIT)}{link_segment}__run_{run_fingerprint}"
)
checkpoint_root = os.path.join(CKPT_DIR, "qwen_e2e", _safe_segment(run_segment))
os.makedirs(checkpoint_root, exist_ok=True)
manifest = {
    "run_fingerprint": run_fingerprint,
    "run_identity": run_identity,
    "extract_cache": _EXTRACT_CACHE_PATH,
    "extract_cache_identity": extract_cache_identity,
    "link_cache": _LINK_CACHE_PATH,
    "link_cache_identity": link_cache_identity,
    "checkpoint_root": checkpoint_root,
}
manifest_path = os.path.join(checkpoint_root, "run_manifest.json")
with open(manifest_path, "w") as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)

def _notebook_qwen_chat(prompt: str) -> str:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=LME_E2E_MAX_NEW_TOKENS, do_sample=False)
    return llm_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _chat_factory():
    return _notebook_qwen_chat

extract_cached_chat = CachedChat(
    Path(_EXTRACT_CACHE_PATH),
    _chat_factory,
    namespace=extract_cache_identity,
    flush_every=FLUSH_EVERY,
    legacy_keys=LEGACY_CACHE_KEYS,
)
if extract_cached_chat.unversioned_cache:
    print("WARNING: extraction cache is legacy md5(prompt)-only; do not cite as a new reproducible paper run.")
fact_extract_fn = build_fact_extract_fn(
    extract_cached_chat,
    prompt_template=FACT_EXTRACTION_PROMPTS[EXTRACT_PROMPT],
)

link_cached_chat = None
slot_link_fn = None
if SLOT_LINKER == "deterministic":
    slot_link_fn = build_slot_link_fn()
elif SLOT_LINKER == "qwen":
    link_cached_chat = CachedChat(
        Path(_LINK_CACHE_PATH),
        _chat_factory,
        namespace=link_cache_identity,
        flush_every=FLUSH_EVERY,
        legacy_keys=LEGACY_CACHE_KEYS,
    )
    if link_cached_chat.unversioned_cache:
        print("WARNING: slot-link cache is legacy md5(prompt)-only; do not cite as a new reproducible paper run.")
    slot_link_fn = build_slot_link_fn(link_cached_chat, confidence_threshold=LINK_THRESHOLD)

print(f"extract prompt: {EXTRACT_PROMPT}; max tokens: {LME_E2E_MAX_NEW_TOKENS}")
print("baselines:", BASELINES_6f)
print("run fingerprint:", run_fingerprint)
print("checkpoint root:", checkpoint_root)
print("run manifest:", manifest_path)
print("extraction cache:", _EXTRACT_CACHE_PATH)
if link_cached_chat is not None:
    print("slot-link cache:", _LINK_CACHE_PATH)
elif SLOT_LINKER == "deterministic":
    print("slot linker: deterministic aliases")

# 3) End-to-end knowledge-update run. Extraction happens once at build time
#    (cached), then the cached writes replay through the requested baselines.
result = {"_run_manifest": manifest}
try:
    instances = load_longmemeval(LME_S_PATH, question_type="knowledge-update", limit=LME_E2E_LIMIT)
    print(f"{len(instances)} knowledge-update questions (end-to-end extraction over full haystack)")
    e2e_report = run_longmemeval_e2e(
        instances, fact_extract_fn,
        intent_mode=INTENT_MODE,
        extract_prompt_name=EXTRACT_PROMPT,
        slot_link_fn=slot_link_fn, slot_linker_name=SLOT_LINKER,
        baselines=BASELINES_6f, seeds=SEEDS,
        embeddings=real_embeddings,
        checkpoint_dir=os.path.join(checkpoint_root, "knowledge_update"),
    )
    result["knowledge_update"] = e2e_report

    print(f"\n=== LongMemEval knowledge-update END-TO-END (intent_mode={INTENT_MODE}; mean [95% CI]) ===")
    print(f"{'Method':<8}{'TaskSuccess up':<22}{'Contradiction dn':<22}{'ConstraintViol dn':<22}")
    for m in e2e_report["methods"]:
        d = e2e_report["decisive_metrics"][m]
        def _ci(k):
            x = d[k]; return f"{x['mean']:.1f} [{x['low']:.1f},{x['high']:.1f}]"
        print(f"{m:<8}{_ci('task_success'):<22}{_ci('contradiction_rate'):<22}{_ci('constraint_violations'):<22}")
    print("\nwrite outcomes:", {m: e2e_report["write_outcomes"][m] for m in e2e_report["methods"]})

    # 4) End-to-end abstention run on the _abs split: full haystack extraction,
    #    then score no-final-answer as abstention; support is diagnostic.
    abst_instances = load_longmemeval(LME_S_PATH, question_type=None, abstention=True, limit=LME_ABST_LIMIT)
    print(f"{len(abst_instances)} abstention questions (end-to-end extraction over full haystack)")
    abst_e2e_report = evaluate_abstention_e2e(
        abst_instances, fact_extract_fn,
        intent_mode=INTENT_MODE,
        extract_prompt_name=EXTRACT_PROMPT,
        slot_link_fn=slot_link_fn, slot_linker_name=SLOT_LINKER,
        baselines=BASELINES_6f, seeds=SEEDS,
        embeddings=real_embeddings,
        checkpoint_dir=os.path.join(checkpoint_root, "abstention"),
    )
    result["abstention"] = abst_e2e_report

    print(f"\n=== LongMemEval abstention END-TO-END (intent_mode={INTENT_MODE}; mean [95% CI]) ===")
    print(f"{'Method':<8}{'Abstention up':<22}{'False answer dn':<22}{'Support diag':<22}")
    for m in abst_e2e_report["methods"]:
        d = abst_e2e_report["abstention_metrics"][m]
        def _ci2(k):
            x = d[k]; return f"{x['mean']:.1f} [{x['low']:.1f},{x['high']:.1f}]"
        false_key = 'false_answer_rate' if 'false_answer_rate' in d else 'false_support_or_answer_rate'
        support = _ci2('supporting_response_rate') if 'supporting_response_rate' in d else 'n/a'
        print(f"{m:<8}{_ci2('abstention_accuracy'):<22}{_ci2(false_key):<22}{support:<22}")
    print("\nabstention counts:", {m: abst_e2e_report["counts"][m] for m in abst_e2e_report["methods"]})
    print("abstention write outcomes:", {m: abst_e2e_report["write_outcomes"][m] for m in abst_e2e_report["methods"]})
finally:
    extract_cached_chat.flush()
    if link_cached_chat is not None:
        link_cached_chat.flush()
    manifest["extract_cache_entries"] = len(extract_cached_chat.cache)
    if link_cached_chat is not None:
        manifest["link_cache_entries"] = len(link_cached_chat.cache)
    if "knowledge_update" in result:
        manifest["knowledge_update_examples_fingerprint"] = result["knowledge_update"].get("examples_fingerprint")
    if "abstention" in result:
        manifest["abstention_examples_fingerprint"] = result["abstention"].get("examples_fingerprint")
    with open(manifest_path, "w") as fh:
        json.dump(manifest, fh, indent=2, sort_keys=True)
    print(f"extraction cache entries: {len(extract_cached_chat.cache)} -> {_EXTRACT_CACHE_PATH}")
    if link_cached_chat is not None:
        print(f"slot-link cache entries: {len(link_cached_chat.cache)} -> {_LINK_CACHE_PATH}")

_p = os.path.join(OUTPUT_DIR, "results_longmemeval_e2e.json")
with open(_p, "w") as fh:
    json.dump(result, fh, indent=2, default=str)
print("Saved ->", _p)


### Notes & caveats
- Sections 3–4 are fully offline/deterministic and need no GPU or keys.
- Section 5 (real embeddings) is the semantic retriever (R2) — part of the system, independent of the extractor; keep it for realistic retrieval.
- **Determinism:** the local extractor uses greedy decoding (`do_sample=False`), and the harness reseeds `torch`/`transformers`/`numpy`/`random` per seed inside `run_full_suite`, so each `(method, seed)` arm is reproducible. Report mean ± 95% CI across the 5 seeds (the harness computes CIs + Holm-Bonferroni significance) rather than relying on a single run.
- **Use the LLM extractor + real embeddings for the headline paper numbers** (Section 6b). Do **not** set `deterministic_test_mode=True` for the research run — it swaps in fake hashing embeddings. The mock extractor + `deterministic_test_mode` (Section 4) is the offline *reference*, useful for CI/ablation sanity, not the main result.
- `Qwen/Qwen2.5-14B-Instruct` is loaded in **full bf16 (~28 GB)** so it fits fully on-GPU on a single **A100 40GB** — full precision, no quantization. For the larger 32B model see the commented alternatives in the load cell (4-bit on 40GB, or bf16 on 80GB).
- The in-process `TransformersExtractor` + real embeddings drive the **full research run (Section 6b)**: the model is loaded once and shared across every baseline/ablation/seed. Section 4 is the CPU-only offline reference.
- Full scale (5 seeds × full benchmark × 8 arms) with a 14B model can take **a few hours** on one A100 40GB — reduce `SEEDS` / `PER_CATEGORY` to shorten.
- **Crash-safe / resumable:** results and per-(method, seed) checkpoints are written to Drive (Section 2b). If Colab refreshes or crashes, just re-run the cell — finished work is skipped and the run resumes. Delete `OUTPUT_DIR/checkpoints` to force a clean rerun.
- OCM storage stays in-memory; the only files written are the results JSON and checkpoints under `OUTPUT_DIR`.
- Free the GPU when done: `del llm_model; import gc, torch; gc.collect(); torch.cuda.empty_cache()`.